<a href="https://colab.research.google.com/github/Precious00001/GenerativeAI-101/blob/main/medical-research-assistant/v2.0/Medical_Research_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-google-genai google-generativeai tiktoken langchain-community
!pip install langchain langchain-google-genai google-generativeai tiktoken faiss-cpu huggingface_hub langchain-community langgraph
!pip install -U langchain-groq
!pip install -U langchain-huggingface sentence-transformers
!pip install -U langchain-community faiss-cpu
!pip install tavily-python
!pip install pypdf
!pip install langchain-text-splitters
!pip install duckduckgo-search
!pip install -U ddgs
!pip install langgraph-checkpoint-sqlite

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
import uuid
import time
from google.colab import userdata
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
import json
from datetime import datetime

In [ ]:
#LLM
groq_api_key = userdata.get("groq_api_key")
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=500,
    max_retries=3,  # retry automatically
    request_timeout=60
)

In [ ]:
#Searching tool
search_tool = DuckDuckGoSearchResults()

#Vectorstore (will set in main)
vectorstore = None

In [ ]:
# Path to the SQLite database file
# This file persists on disk so memory survives session restarts
DB_PATH = "medical_research_memory.db"

def initialize_memory_db():
    """Create the memory database and tables if they don't exist."""

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Table for storing research context
    # Tracks topics, papers, and conclusions the researcher has explored
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS research_context (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            topic TEXT NOT NULL,
            details TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)

    # Table for storing open threads
    # Tracks unresolved questions the researcher wants to come back to
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS open_threads (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            question TEXT NOT NULL,
            context TEXT,
            resolved INTEGER DEFAULT 0,
            timestamp TEXT NOT NULL
        )
    """)

    # Table for storing user preferences
    # Tracks how the researcher likes answers structured
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS user_preferences (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            preference_key TEXT UNIQUE NOT NULL,
            preference_value TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)

    conn.commit()
    conn.close()
    print("Memory database initialized!")

# Initialize the database
initialize_memory_db()

In [ ]:
def save_research_context(topic: str, details: str):
    """Save a research topic and its details to memory."""

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Store the topic with a timestamp
    cursor.execute("""
        INSERT INTO research_context (topic, details, timestamp)
        VALUES (?, ?, ?)
    """, (topic, details, datetime.now().isoformat()))

    conn.commit()
    conn.close()


def get_research_context(topic: str) -> str:
    """Retrieve past research context relevant to a topic."""

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Search for any past context that mentions this topic
    cursor.execute("""
        SELECT topic, details, timestamp
        FROM research_context
        WHERE topic LIKE ?
        ORDER BY timestamp DESC
        LIMIT 5
    """, (f"%{topic}%",))

    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return ""

    # Format past context into a readable string
    context_parts = []
    for topic, details, timestamp in rows:
        context_parts.append(f"- [{timestamp[:10]}] {topic}: {details}")

    return "\n".join(context_parts)


def save_open_thread(question: str, context: str = ""):
    """Save an unresolved question to come back to later."""

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO open_threads (question, context, timestamp)
        VALUES (?, ?, ?)
    """, (question, context, datetime.now().isoformat()))

    conn.commit()
    conn.close()


def get_open_threads() -> str:
    """Retrieve all unresolved questions from past sessions."""

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Only get unresolved threads (resolved = 0)
    cursor.execute("""
        SELECT question, context, timestamp
        FROM open_threads
        WHERE resolved = 0
        ORDER BY timestamp DESC
        LIMIT 5
    """)

    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return ""

    # Format open threads into a readable string
    threads = []
    for question, context, timestamp in rows:
        threads.append(f"- [{timestamp[:10]}] {question}")

    return "\n".join(threads)


def save_user_preference(key: str, value: str):
    """Save or update a user preference."""

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Insert or replace if key already exists
    cursor.execute("""
        INSERT OR REPLACE INTO user_preferences (preference_key, preference_value, timestamp)
        VALUES (?, ?, ?)
    """, (key, value, datetime.now().isoformat()))

    conn.commit()
    conn.close()


def get_user_preferences() -> str:
    """Retrieve all saved user preferences."""

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT preference_key, preference_value
        FROM user_preferences
    """)

    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return ""

    # Format preferences into a readable string
    prefs = [f"- {key}: {value}" for key, value in rows]
    return "\n".join(prefs)
def auto_save_open_thread(question: str, answer: str):
    """Automatically save a question as an open thread if the answer is incomplete.
    Called after every agent response to check if the question was fully resolved.
    """
    # Keywords that suggest the answer was incomplete
    incomplete_signals = [
        "i don't know",
        "i'm not sure",
        "unclear",
        "cannot find",
        "no information",
        "not available",
        "please upload",
        "no documents loaded",
        "unable to",
        "insufficient"
    ]

    # Check if any incomplete signal is in the answer
    answer_lower = answer.lower()
    is_incomplete = any(signal in answer_lower for signal in incomplete_signals)

    if is_incomplete:
        # Save as open thread for later
        save_open_thread(
            question=question,
            context=f"Partial answer received: {answer[:200]}"
        )
        print(f"📌 Open thread saved: '{question}'")
    else:
        # Answer was complete — mark as resolved if it was an open thread
        mark_thread_resolved(question)

def mark_thread_resolved(question: str):
    """Automatically save a question as an open thread if the answer is incomplete,
    or mark it as resolved if the answer is complete.
    Called after every agent response to check if the question was fully resolved.
    """
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Find and mark the matching thread as resolved
    cursor.execute("""
        UPDATE open_threads
        SET resolved = 1
        WHERE question LIKE ?
    """, (f"%{question[:50]}%",))

    conn.commit()

    # Check if anything was updated
    if cursor.rowcount > 0:
        print(f"✅ Thread resolved: '{question[:50]}'")

    conn.close()

In [ ]:
def load_pdfs(pdf_paths):
  all_text = ""
  for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents = loader.load()
    for doc in documents:
        all_text += doc.page_content + "\n"
    print(f"Loaded: {path}")
  return all_text


def chunk_text(text, chunk_size=500):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunk = text[i:i+chunk_size]
    if len(chunk) > 200:
      chunks.append(chunk)
  return chunks or [text]

# Load embeddings once at the top level, not inside the function
print("Loading embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings model loaded!")

def create_vector_store(texts):
  # Convert raw text chunks into LangChain Document objects
    documents = [
        Document(
            page_content=text,
            metadata={"id": str(uuid.uuid4())}
        ) for text in texts
    ]

    # Use the already loaded embeddings instead of reloading
    vector_store = FAISS.from_documents(documents, embeddings)

    return vector_store

In [ ]:
@tool
def search_documents(query: str) -> str:
  """Search for relevant information from the loaded PDF documents."""

  #Check if vector store has been created
  if vectorstore is None:
    return "NO PDF LOADED — Source: Internet Search fallback not available. Please upload a PDF."

  #Search for similar chunks in vector store
  docs_with_scores = vectorstore.similarity_search_with_score(query, k=3)

  # Filter out chunks that are not relevant enough
  # Lower score = more similar, so we keep score below 1.5
  relevant_docs = [doc for doc, score in docs_with_scores if score <1.5]

  if not relevant_docs:
    return "No relevant docs found."

  #combine relevant chunks into a single string
  combined = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])
  result = f"Source: PDF Document\n\nRelevant documents:\n{combined}"

  # --- Memory: save this research topic silently ---
  save_research_context(topic=query, details=combined[:300])

  return result


@tool
def dosage_calculator(weight_and_dose: str) -> str:
  """ Calculate total dosage given patient weight and dose per kg.
  Input format: 'wieght_kg, dose_per_kg, e.g. '70, 5'
  """
  try:
    #split the input string into weight and dose
    parts = weight_and_dose.split(",")
    weight = float(parts[0].strip())
    dose_per_kg = float(parts[1].strip())

    #Calculate tottal dosa
    total_dose = weight * dose_per_kg

    return f"Source: Dosage Calculator\n\nTotal dosage: {total_dose}mg (Weight: {weight}kg × Dose: {dose_per_kg}mg/kg)"
  except Exception as e:
    return f"Error calculating dosage: {str(e)}"

@tool
def search_drug_interactions(drugs: str) -> str:
    """Search for interactions between specified drugs.
    Input format: 'drug1, drug2' e.g. 'aspirin, ibuprofen'
    """
    try:
        # DuckDuckGo returns a single string directly
        results = search_tool.invoke({"query": f"drug interactions between {drugs}"})
        return f"Source: Internet Search\n\n{results}"
    except Exception as e:
        return f"Error searching drug interactions: {str(e)}"

In [ ]:
system_prompt = """You are a medical research assistant. You ONLY have access to these EXACT tools:
1. search_documents
2. dosage_calculator
3. search_drug_interactions

STRICT RULES:
1. NEVER call any tool not listed above
2. NEVER use brave_search, google_search or any other search tool
3. Use ONE tool only once per question
4. After getting the tool result, give your final answer immediately
5. Do NOT keep reasoning after you have an answer
6. ALWAYS format your final answer EXACTLY like this:

Answer: [your answer here]
Source: [where the information came from]

Never skip the Source line. Never skip the Answer line
7. If a tool returns 'No documents loaded', your source MUST be 'No PDF loaded' — never say 'PDF Documents'."""

def main():
    global vectorstore

    # --- Load and index PDF documents FIRST ---
    try:
        pdf_paths = ["/content/Introduction-to-research-and-ethics-Iveta-6-Feb.pdf"]
        pdf_text = load_pdfs(pdf_paths)
        chunks = chunk_text(pdf_text)
        chunks = chunks[:20] # limit during dev to avoid rate limits
        vectorstore = create_vector_store(chunks)
        print("Vector store created successfully!")
    except Exception as e:
        print(f"Vector store creation failed: {str(e)}")
        vectorstore = None


    # --- Load past memory before starting ---
    past_research = get_research_context("")
    past_threads = get_open_threads()
    past_preferences = get_user_preferences()

    # --- Build memory context string to silently inform the agent ---
    memory_context = ""
    if past_research:
        memory_context += f"\nPast research topics this user has explored:\n{past_research}"
    if past_threads:
        memory_context += f"\nUnresolved questions from past sessions:\n{past_threads}"
    if past_preferences:
        memory_context += f"\nUser preferences:\n{past_preferences}"

    # --- Wire SqliteSaver for conversation persistence ---
    with SqliteSaver.from_conn_string(DB_PATH) as checkpointer:
      # --- Set up tools and agent AFTER vector store ---
      tools = [search_documents, dosage_calculator, search_drug_interactions]
      agent = create_react_agent(llm, tools, prompt=system_prompt)

      # --- Thread config — identifies this user's conversation ---
      config = {
            "configurable": {"thread_id": "medical_researcher_1"},
            "recursion_limit": 15
        }


      # --- Test questions ---
      questions = [
          "What should I do before I start research?",
          "What is the dosage for a 70kg patient at 5mg/kg?",
          "What are the interactions between aspirin and ibuprofen?"
      ]

      # --- Run each question through the agent ---
      for question in questions:
          print(f"\nQuestion: {question}")

          # Inject memory context into the question silently
          full_input = question
          if memory_context:
              full_input = f"{question}\n\n[Context from past sessions:{memory_context}]"
          response = agent.invoke(
              {"messages": [("human", question)]},
              config={"recursion_limit": 15}
          )
          # Store answer in variable instead of printing directly
          answer = response['messages'][-1].content
          print(answer)
          print("-" * 50)

          # Auto-save as open thread if answer was incomplete
          auto_save_open_thread(question, answer)

          time.sleep(2)
main()